In [1]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [2]:
from pathlib import Path
import polars as pl
from IPython.display import display

# SRC Data

In [25]:
data_dir = "/home/chemi_t/Documents/sci_fi/Programming/Job/Ameriabank"
tx_path_str = f"{data_dir}/detail/trx/**/*.parquet"

In [26]:
! ls {data_dir}/detail/trx/**/*.parquet

'/home/chemi_t/Documents/sci_fi/Programming/Job/Ameriabank/detail/trx/fold=0/part-00001-f1f2463d-80e2-4335-8981-4fc9e73f56f4.c000.snappy.parquet'
'/home/chemi_t/Documents/sci_fi/Programming/Job/Ameriabank/detail/trx/fold=0/part-00085-f1f2463d-80e2-4335-8981-4fc9e73f56f4.c000.snappy.parquet'
'/home/chemi_t/Documents/sci_fi/Programming/Job/Ameriabank/detail/trx/fold=0/part-00086-f1f2463d-80e2-4335-8981-4fc9e73f56f4.c000.snappy.parquet'
'/home/chemi_t/Documents/sci_fi/Programming/Job/Ameriabank/detail/trx/fold=0/part-00087-f1f2463d-80e2-4335-8981-4fc9e73f56f4.c000.snappy.parquet'
'/home/chemi_t/Documents/sci_fi/Programming/Job/Ameriabank/detail/trx/fold=0/part-00088-f1f2463d-80e2-4335-8981-4fc9e73f56f4.c000.snappy.parquet'
'/home/chemi_t/Documents/sci_fi/Programming/Job/Ameriabank/detail/trx/fold=0/part-00089-f1f2463d-80e2-4335-8981-4fc9e73f56f4.c000.snappy.parquet'
'/home/chemi_t/Documents/sci_fi/Programming/Job/Ameriabank/detail/trx/fold=0/part-00090-f1f2463d-80e2-4335-8981-4fc9e73f56f4

In [27]:
dates = [
    "2020-01-01 13:45:48",
    "2020-01-01 16:42:13",
    "2020-01-01 16:45:09",
    "2020-01-02 18:12:48",
    "2020-01-03 19:45:32",
    "2020-01-08 23:16:43",
]
df = pl.DataFrame({"dt": dates, "a": [3, 7, 5, 9, 2, 1]}).with_columns(
    pl.col("dt").str.strptime(pl.Datetime).set_sorted()
)
df.sort("dt")
lf = df.lazy().sort("dt")
out = lf.rolling(index_column="dt", period="2d").agg(
    [
        pl.sum("a").alias("sum_a"),
        pl.min("a").alias("min_a"),
        pl.max("a").alias("max_a"),
    ]
)

In [28]:
lf_tx = pl.scan_parquet(tx_path_str)

# trx profiling

In [29]:
lf_tx.head(n=5).collect()

client_id,event_time,amount,event_type,event_subtype,currency,src_type11,src_type12,dst_type11,dst_type12,src_type21,src_type22,src_type31,src_type32
str,datetime[ns],f32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""01a1aca1531e0d86b605759fedd36e…",2022-10-26 13:58:59.938408,1.0215e6,16,18,11,22,47,780,21619,21490,83,364,58
"""01a1aca1531e0d86b605759fedd36e…",2021-02-09 07:43:20.639579,300765.5625,1,12,11,4,958,306,26766,21490,83,364,58
"""01a1aca1531e0d86b605759fedd36e…",2022-10-22 09:12:06.614108,606154.9375,16,18,11,22,47,780,21619,21490,83,364,58
"""01a1aca1531e0d86b605759fedd36e…",2021-02-12 15:12:20.660481,80325.914062,1,12,11,4,958,306,26766,21490,83,364,58
"""01a1aca1531e0d86b605759fedd36e…",2022-10-08 06:52:28.877461,122286.59375,16,18,11,22,47,780,21619,21490,83,364,58


In [30]:
max_min_date = lf_tx.select(
    pl.col("event_time").dt.date().max().alias("max_date"),
    pl.col("event_time").dt.date().min().alias("min_date"),
).collect()
date_range = max_min_date["max_date"].item() - max_min_date["min_date"].item()

In [31]:
n_clients = lf_tx.select(pl.col("client_id").n_unique()).collect()
display(n_clients)

client_id
u32
98721


In [ ]:
q = lf_tx.select(pl.col("client_id").value_counts())
q = (
    q.collect()
    .select(pl.col("client_id").struct.field("count"))
    .filter(pl.col("count") < 10000)
)
display(q.select(pl.len()))
q["count"].hist(bin_count=10).sort("breakpoint")

len
u32
98708


breakpoint,category,count
f64,cat,u32
998.3,"""[1.0, 998.3]""",88697
1995.6,"""(998.3, 1995.6]""",8508
2992.9,"""(1995.6, 2992.9]""",1201
3990.2,"""(2992.9, 3990.2]""",191
4987.5,"""(3990.2, 4987.5]""",70
5984.8,"""(4987.5, 5984.8]""",23
6982.1,"""(5984.8, 6982.1]""",4
7979.4,"""(6982.1, 7979.4]""",4
8976.7,"""(7979.4, 8976.7]""",6


# Proto asset ops

In [33]:
lf_tx_part = lf_tx.filter(pl.col("client_id").cast(pl.String).hash() % 100 == 0)
n_client_in_part = lf_tx_part.select(pl.col("client_id").n_unique()).collect()
n_date_part = lf_tx_part.select(pl.col("event_time").dt.date().n_unique()).collect()
print(n_client_in_part)
print(n_date_part)

shape: (1, 1)
┌───────────┐
│ client_id │
│ ---       │
│ u32       │
╞═══════════╡
│ 986       │
└───────────┘
shape: (1, 1)
┌────────────┐
│ event_time │
│ ---        │
│ u32        │
╞════════════╡
│ 731        │
└────────────┘


In [34]:
lf_tx_part = lf_tx.filter(pl.col("client_id").cast(pl.String).hash() % 100 == 0)
n_client_in_part = lf_tx_part.select(pl.col("client_id").n_unique())
print(n_client_in_part)
out = (
    lf_tx_part.sort("event_time")
    .select(pl.col("event_time").dt.date(), pl.col("client_id"), pl.col("amount"))
    .rolling(index_column="event_time", period="1w", group_by="client_id")
    .agg(
        [
            pl.sum("amount").alias("sum_a"),
        ]
    )
    .collect()
)

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

SELECT [col("client_id").n_unique()]
  FILTER [([(col("client_id").hash()) % (100)]) == (0)]
  FROM
    Parquet SCAN [/home/chemi_t/Documents/sci_fi/Programming/Job/Ameriabank/detail/trx/fold=0/part-00001-f1f2463d-80e2-4335-8981-4fc9e73f56f4.c000.snappy.parquet, ... 399 other sources]
    PROJECT */14 COLUMNS
    ESTIMATED ROWS: 3154000


In [35]:
out.select(pl.col("client_id"), pl.col("event_time").dt.date().alias("date")).group_by(
    "client_id"
).agg(pl.col("date").len())

client_id,date
str,u32
"""e436b34256bd73d6c2c70999804c28…",662
"""c1ec61ceab7d7d055559fde884fcad…",16
"""aa3e88d254520d4532b6ef1c1c35fb…",2
"""3c6f299fbfd77ce5258b746b7c02e9…",3
"""a96537c14bb64087b3c0519c0cf9d2…",242
…,…
"""3a698d88313a1cd29741d1607f3189…",447
"""0f2601af00b819159b2e61b1c4cb66…",707
"""e6b028357bae5c165a910a78131cdf…",1202


In [ ]:
date_range.days

AttributeError: 'datetime.timedelta' object has no attribute 'dt'

In [38]:
out.select(pl.len())

len
u32
367964
